<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [4]</a>'.</span>

## Importar API da Base dos Dados



In [1]:
!pip install basedosdados

## Controle de Versão

In [2]:
# --- TÓPICO 0: CONFIGURAÇÕES DE AMBIENTE E REPRODUTIBILIDADE ---
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import pandas_gbq
import basedosdados
import matplotlib
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_utils import (
    carregar_snis,
    preparar_populacao_referencia,
    calcular_flags_evidencia,
    limpar_df_gold,
    processar_saude,
    integrar_saude_tabnet,
    calcular_risco_social_final,
    validar_populacao,
    classificar_qualidade_populacao,
    analisar_municipio,
 )

# 1. Registro de Versões (Garante que você saiba em qual ambiente o código funcionou)
print(f"📌 Pandas: {pd.__version__}")
print(f"📌 Numpy: {np.__version__}")
print(f"📌 Pandas-GBQ: {pandas_gbq.__version__}")
print(f"📌 Matplotlib: {matplotlib.__version__}") 


# 2. Comando para gerar o requirements.txt (Rode se quiser exportar para outro ambiente)
# !pip freeze > requirements.txt

# 3. Configurações Globais do Pandas (Para evitar comportamentos inesperados)
try:
    pd.set_option('future.no_silent_downcasting', True)
except Exception:
    print("INFO: Opcao future.no_silent_downcasting nao existe nesta versao do pandas.")

📌 Pandas: 2.3.3
📌 Numpy: 2.4.4
📌 Pandas-GBQ: 0.35.0
📌 Matplotlib: 3.10.9


## Autenticar usuário

In [3]:
import pandas_gbq

try:
    from google.colab import auth
except ModuleNotFoundError:
    print("INFO: Google Colab nao detectado; pulando autenticacao.")
else:
    auth.authenticate_user()
    print("Autenticado com sucesso!")

INFO: Google Colab nao detectado; pulando autenticacao.


# Saneamento: Água e Esgoto

## Importar Base de Dados

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [4]:
# Seu ID corrigido (garanta que não haja espaços antes ou depois)
import pandas_gbq

PROJECT_ID = "analise-saneamento"

# SQL otimizada: seleciona apenas colunas usadas e filtra UF/ano na origem
sql = """
SELECT
  ano,
  id_municipio,
  sigla_uf,
  quantidade_economia_residencial_ativa_agua,
  quantidade_economia_residencial_ativa_esgoto,
  quantidade_ligacao_total_agua,
  quantidade_ligacao_total_esgoto,
  populacao_urbana,
  populacao_atendida_agua,
  indice_atendimento_total_agua,
  indice_atendimento_esgoto_agua,
  indice_atendimento_urbano_agua,
  indice_tratamento_esgoto,
  indice_perda_distribuicao_agua,
  indice_consumo_agua_per_capita,
  volume_esgoto_coletado,
  volume_esgoto_tratado,
  extensao_rede_agua,
  extensao_rede_esgoto,
  populacao_atentida_esgoto AS populacao_urbana_atendida_esgoto,
  quantidade_ligacao_ativa_esgoto,
  investimento_total_municipio,
  investimento_total_estado,
  investimento_total_prestador,
  despesa_exploracao,
  arrecadacao_total,
  receita_operacional
FROM `basedosdados.br_mdr_snis.municipio_agua_esgoto`
WHERE sigla_uf = 'ES' AND ano >= 2006
"""

try:
    df = pandas_gbq.read_gbq(sql, project_id=PROJECT_ID)
    print("Sucesso! Dados do ES carregados.")
    display(df.head())
except Exception as e:
    raise RuntimeError(f"Erro ao acessar a tabela de saneamento: {e}")


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&state=U3HdDYmmACtmgYZRhQpY7nF7LLfeAt&code_challenge=9e-aLkNfQwBTritNZBUT8KcW9hTMPvhyjrRzi8985QM&code_challenge_method=S256&prompt=consent&access_type=offline


RuntimeError: Erro ao acessar a tabela de saneamento: Reason: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/analise-saneamento/queries?prettyPrint=false: Access Denied: Project analise-saneamento: User does not have bigquery.jobs.create permission in project analise-saneamento.

## Definindo estrutura do data frame

In [ ]:
colunas_selecionadas = [
    # Identificação
    'ano', 'id_municipio', 'sigla_uf',

    # Para identificar comunidades vulneráveis
    'quantidade_economia_residencial_ativa_agua',
    'quantidade_economia_residencial_ativa_esgoto',
    'quantidade_ligacao_total_agua',
    'quantidade_ligacao_total_esgoto',

    # Demografia e Cobertura
    'populacao_urbana',
    'populacao_atendida_agua',
    'indice_atendimento_total_agua',
    'indice_atendimento_esgoto_agua',
    'indice_atendimento_urbano_agua',

    # Qualidade e Performance
    'indice_tratamento_esgoto',
    'indice_perda_distribuicao_agua',
    'indice_consumo_agua_per_capita',
    'volume_esgoto_coletado',
    'volume_esgoto_tratado',

    # Infraestrutura (EVIDÊNCIAS PARA O SCRIPT)
    'extensao_rede_agua',
    'extensao_rede_esgoto',
    'populacao_urbana_atendida_esgoto',
    'quantidade_ligacao_ativa_esgoto',

    # Investimentos
    'investimento_total_municipio',
    'investimento_total_estado',
    'investimento_total_prestador',
    'despesa_exploracao',
    'arrecadacao_total',
    'receita_operacional'
]

df_silver = df[colunas_selecionadas].copy()

# Asserção para garantir que o filtro do SQL foi aplicado
assert df_silver['ano'].min() >= 2006, "Dado fora do intervalo esperado — verificar SQL"
print(f"Novo tamanho do dataset: {df_silver.shape}")


## Limpeza dos Dados

In [ ]:
# --- TRATAMENTO DE POPULAÇÃO E ESGOTO ---
df_gold = limpar_df_gold(df_silver)

print("Processamento final concluído! Base limpa com interpolação conservadora.")
print("✅ População de referência preparada para o cálculo de morbidade.")

## Integração de Dados de Saúde (TabNet)

In [ ]:
# --- CÉLULA 7: INTEGRAÇÃO DATASUS E CÁLCULO DE PESOS EPIDEMIOLÓGICOS ---

# Pesos versionados para evitar deriva histórica silenciosa no dashboard
VERSAO_PESOS = '2025-04'
W_AGUA_CALCULADO = 0.52
W_ESGOTO_CALCULADO = 0.48
RECALCULAR_PESOS = False  # mude para True apenas ao atualizar os CSVs

df_final, W_AGUA, W_ESGOTO = integrar_saude_tabnet(
    df_gold,
    str(PROJECT_ROOT / "data" / "raw" / "saude_agua_es.csv"),
    str(PROJECT_ROOT / "data" / "raw" / "saude_esgoto_es.csv"),
    versao_pesos=VERSAO_PESOS,
    w_agua_calculado=W_AGUA_CALCULADO,
    w_esgoto_calculado=W_ESGOTO_CALCULADO,
    recalcular_pesos=RECALCULAR_PESOS
 )

In [ ]:
# Qual é o nome real da variável retornada pela integração?
# Substitua df_saude_integrado pelo nome real que você usa
print("Colunas do resultado da integração de saúde:")
print([c for c in df_final.columns if df_final[c].notna().any()])

In [ ]:
# Verificar se a taxa existe com outro nome
candidatas = [c for c in df_final.columns 
              if df_final[c].notna().any() 
              and df_final[c].dtype in ['float64', 'int64']
              and df_final[c].max() > 1]
print("Colunas numéricas com dados:")
print(candidatas)

## Engenharia de Risco Social Multidimensional (Água vs. Esgoto)

In [ ]:
# --- CÉLULA 8: ENGENHARIA DO RISCO SOCIAL FINAL ---
df_final = calcular_risco_social_final(df_final, W_AGUA, W_ESGOTO)

print("💎 Risco Social Final calculado (modelo robusto)")
print("📊 Escala: 0 a 100")
print("🧠 Proteções: ausência de dados + controle de escala + fallback estrutural")
print("🔎 Flags de truncamento em eficiencia_arrecadacao registradas para auditoria")

# Comunidades vulneráveis

## Diagnóstico de Vetores Dominantes

In [ ]:

# --- DIAGNÓSTICO VETORIZADO DO VETOR DOMINANTE (ROBUSTO A NaN) ---

if 'internacoes_agua' in df_final.columns:
    internacoes_agua = pd.Series(
        pd.to_numeric(df_final['internacoes_agua'], errors='coerce'),
        index=df_final.index
    )
else:
    internacoes_agua = pd.Series(np.nan, index=df_final.index)

if 'internacoes_esgoto' in df_final.columns:
    internacoes_esgoto = pd.Series(
        pd.to_numeric(df_final['internacoes_esgoto'], errors='coerce'),
        index=df_final.index
    )
else:
    internacoes_esgoto = pd.Series(np.nan, index=df_final.index)

if 'tem_dado_saude' in df_final.columns:
    tem_dado_saude = df_final['tem_dado_saude'].fillna(False)
else:
    tem_dado_saude = internacoes_agua.notna() | internacoes_esgoto.notna()

# Sem dado segue apenas a flag consolidada (internações já chegam inteiras da Célula 19).
sem_dados = ~tem_dado_saude
ambos_zero = (~sem_dados) & (internacoes_agua == 0) & (internacoes_esgoto == 0)
vetor_agua = (~sem_dados) & (internacoes_agua > internacoes_esgoto)
empate = (~sem_dados) & (internacoes_agua == internacoes_esgoto) & (internacoes_agua > 0)
vetor_esgoto = (~sem_dados) & (internacoes_agua < internacoes_esgoto)

condicoes = [
    sem_dados,
    ambos_zero,
    vetor_agua,
    empate,
    vetor_esgoto
]

escolhas = [
    'Sem Dados',
    'Baixo Impacto',
    'Vetor Água',
    'Empate',
    'Vetor Esgoto'
]

df_final['vetor_dominante_doenca'] = np.select(condicoes, escolhas, default='Inconsistente')

print('📊 Perfil epidemiológico vetorizado (ES 2022):')
print(df_final[df_final['ano'] == 2022]['vetor_dominante_doenca'].value_counts())
print("⚠️ Registros marcados como 'Inconsistente':", (df_final['vetor_dominante_doenca'] == 'Inconsistente').sum())

# Validações

In [ ]:
# --- VALIDAÇÃO DA POPULAÇÃO DE REFERÊNCIA ---

def validar_populacao(df):
    df = df.sort_values(['id_municipio', 'ano']).copy()

    relatorio = {}

    # 1. Valores inválidos
    invalidos = df[
        (df['populacao_ref'].isna()) |
        (df['populacao_ref'] <= 0)
    ]
    relatorio['valores_invalidos'] = invalidos

    # 2. Crescimento ano a ano (%)
    df['populacao_anterior'] = df.groupby('id_municipio')['populacao_ref'].shift(1)
    df['crescimento_pct'] = (
        (df['populacao_ref'] - df['populacao_anterior']) /
        df['populacao_anterior']
    ) * 100

    crescimento_absurdo = df[
        df['crescimento_pct'].abs() > 20
    ]
    relatorio['crescimento_absurdo'] = crescimento_absurdo

    # 3. Outliers (z-score por município)

    df['outlier'] = df.groupby('id_municipio', group_keys=False)['populacao_ref'].transform(
        lambda s: ((s - s.mean()) / s.std()).abs().gt(3) if pd.notna(s.std()) and s.std() != 0 else pd.Series(False, index=s.index)
    )
    relatorio['outliers'] = df[df['outlier']]

    # 4. Saltos absolutos grandes
    df['delta_abs'] = (df['populacao_ref'] - df['populacao_anterior']).abs()

    salto_grande = df[
        df['delta_abs'] > df['populacao_ref'] * 0.15
    ]
    relatorio['saltos_grandes'] = salto_grande

    # 5. Lacunas originalmente imputadas
    if 'populacao_ref_era_nula' in df.columns:
        relatorio['valores_imputados'] = df[df['populacao_ref_era_nula']]
    else:
        relatorio['valores_imputados'] = df.iloc[0:0].copy()

    print("\n📊 RELATÓRIO DE VALIDAÇÃO DA POPULAÇÃO\n")
    print(f"❌ Valores inválidos: {len(relatorio['valores_invalidos'])}")
    print(f"📈 Crescimentos suspeitos (>20%): {len(relatorio['crescimento_absurdo'])}")
    print(f"📊 Outliers estatísticos: {len(relatorio['outliers'])}")
    print(f"⚠️ Saltos absolutos grandes: {len(relatorio['saltos_grandes'])}")
    print(f"🩹 Valores imputados/interpolados: {len(relatorio['valores_imputados'])}")

        # Apelidos e coluna adicional para células de classificação posteriores usarem sem recalcular
    df['pop_ant'] = df['populacao_anterior']
    df['pop_prox'] = df.groupby('id_municipio')['populacao_ref'].shift(-1)

    return relatorio, df

# Valida a base que será efetivamente exportada
relatorio, df_validado = validar_populacao(df_final)

In [ ]:
# municípios mais problemáticos
relatorio['crescimento_absurdo']['id_municipio'].value_counts().head(10)

In [ ]:
relatorio['outliers']['id_municipio'].value_counts().head(10)

In [ ]:
relatorio['saltos_grandes']['id_municipio'].value_counts().head(10)

In [ ]:
def analisar_municipio(df, municipio_id):

    df_m = df[df['id_municipio'].astype(str) == str(municipio_id)].sort_values('ano')

    print(df_m[['ano', 'populacao_ref', 'crescimento_pct']])

    plt.plot(df_m['ano'], df_m['populacao_ref'], marker='o')
    plt.title(f"Município {municipio_id}")
    plt.grid()
    plt.show()


In [ ]:
analisar_municipio(df_validado, '3200359')

In [ ]:
df_validado, df_final = classificar_qualidade_populacao(df_validado, df_final)

print('✅ Flags de qualidade da população sincronizadas em df_final.')

In [ ]:
df_validado['tipo_erro'].value_counts()

In [ ]:
df_validado[df_validado['tipo_erro'] != 'ok'][
    ['id_municipio', 'ano', 'populacao_ref', 'tipo_erro']
]

In [ ]:
df_validado[df_validado['tipo_erro'] == 'mudanca_base']['id_municipio'].unique()

In [ ]:
# Cole isso logo ANTES da célula de exportação do parquet
print("Taxa_Morbidade no momento da exportação:")
print(f"  Não-nulos: {df_final['Taxa_Morbidade_100k_Hab'].notna().sum()}")
print(f"  Nulos: {df_final['Taxa_Morbidade_100k_Hab'].isna().sum()}")

# Exportação

In [ ]:
# --- EXPORTAÇÃO DE ALTA PERFORMANCE (PARQUET) ---

# O formato Parquet preserva os tipos de dados (int, float, datetime)
# e oferece compressão superior ao CSV, ideal para o consumo no Streamlit.

try:
    PROCESSED = PROJECT_ROOT / "data" / "processed"
    PROCESSED.mkdir(parents=True, exist_ok=True)

    df_final.to_parquet(PROCESSED / "base_diamante_es_vfinal.parquet", index=False)
    print(f"🚀 Projeto finalizado com sucesso!")
    print(f"📦 Arquivo 'base_diamante_es_vfinal.parquet' gerado.")
    print(f"📊 Total de registros: {df_final.shape[0]}")
    print(f"💾 Tipos preservados: Internações permanecem como INT, Índices como FLOAT.")
except Exception as e:
    print(f"❌ Erro ao exportar para Parquet: {e}")
    # Fallback de segurança para CSV caso o ambiente falte dependência (pyarrow/fastparquet)
    df_final.to_csv('base_diamante_es_vfinal.csv', sep=';', index=False, encoding='utf-8-sig')
    print("⚠️ Exportado para CSV como alternativa de segurança.")


In [ ]:
# Diagnóstico da integração de saúde
print("1. Colunas de flags de saúde:")
print(df_final[['tem_dado_saude_agua', 'tem_dado_saude_esgoto', 'tem_dado_saude']].sum())

print("\n2. Coluna vetor_dominante_doenca (primeiras linhas não-nulas):")
print(df_final['vetor_dominante_doenca'].dropna().head(10))

print("\n3. Verificar se o arquivo de saúde foi carregado:")
# Troque pelo nome real da variável do seu df de saúde bruto
# Provavelmente algo como df_saude, df_tabnet, df_morbidade
# Se não souber, rode:
print([v for v in dir() if 'saude' in v.lower() or 'tabnet' in v.lower() or 'morbi' in v.lower()])